<a href="https://colab.research.google.com/github/R1ng1/2nd_M.Sc_Final_Project/blob/main/baseline_ner_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# create a project folder on your Drive
!mkdir -p /content/drive/MyDrive/iltur_ner_baseline
%cd /content/drive/MyDrive/iltur_ner_baseline


Mounted at /content/drive
/content/drive/MyDrive/iltur_ner_baseline


In [ ]:
# run in a Colab cell (prefix !)
!pip install -q transformers datasets accelerate seqeval huggingface_hub
# optional but useful: tokenizers, sentencepiece if you plan multilingual models
!pip install -q tokenizers sentencepiece


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.6/221.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.3/377.3 kB 34.9 MB/s eta 0:00:00


In [ ]:
# only if you want to push models/large downloads
!pip install -q huggingface_hub
from huggingface_hub import login
# then run: login("YOUR_HF_TOKEN")  # paste your token when prompted


In [ ]:
from datasets import load_dataset

# Show that datasets package is working
print("datasets version loaded.")

# Load the IL-TUR dataset (this will fetch from Hugging Face)
# Use token=True to access gated datasets if you have logged in with huggingface_hub.login() or have the HUGGING_FACE_HUB_TOKEN env var set.
ds = load_dataset("Exploration-Lab/IL-TUR", "lner", token=True)
print(ds)   # prints dataset dict and available configs/splits
# If the dataset has configs you may see something like ds['l-ner'] or ds['L-NER']

datasets version loaded.


README.md:   0%|          | 0.00/37.9k [00:00<?, ?B/s]

lner/fold_1-00000-of-00001.parquet:   0%|          | 0.00/584k [00:00<?, ?B/s]

lner/fold_2-00000-of-00001.parquet:   0%|          | 0.00/589k [00:00<?, ?B/s]

lner/fold_3-00000-of-00001.parquet:   0%|          | 0.00/612k [00:00<?, ?B/s]

Generating fold_1 split:   0%|          | 0/35 [00:00<?, ? examples/s]

Generating fold_2 split:   0%|          | 0/35 [00:00<?, ? examples/s]

Generating fold_3 split:   0%|          | 0/35 [00:00<?, ? examples/s]

DatasetDict({
    fold_1: Dataset({
        features: ['id', 'text', 'spans'],
        num_rows: 35
    })
    fold_2: Dataset({
        features: ['id', 'text', 'spans'],
        num_rows: 35
    })
    fold_3: Dataset({
        features: ['id', 'text', 'spans'],
        num_rows: 35
    })
})


The `LocalTokenNotFoundError` indicates that the Hugging Face token required to access the gated dataset was not found.

To fix this, you need to provide your Hugging Face token. You can do this by:

1.  **Logging in interactively:** Run `huggingface_hub.login()` in a code cell and paste your token when prompted.
2.  **Setting an environment variable:** Set the `HUGGING_FACE_HUB_TOKEN` environment variable with your token.

Since the dataset loading code uses `token=True`, the `datasets` library will automatically look for the token in these locations.

In [ ]:
# Uncomment the line below and run this cell to log in to Hugging Face
# from huggingface_hub import login
# login()

# Step-2

In [ ]:
# inspect one sample from the first fold
sample = ds['fold_1'][0]
print("Text:\n", sample['text'])
print("\nSpans:\n", sample['spans'])


Text:
 REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, entered the land where the complainant was working with his sons an

In [ ]:
# See the span structure and possible labels
for s in ds['fold_1'][0]['spans']:
    print(s)


{'start': 137, 'end': 153, 'label': 1}
{'start': 252, 'end': 261, 'label': 10}
{'start': 276, 'end': 308, 'label': 7}
{'start': 359, 'end': 369, 'label': 7}
{'start': 575, 'end': 585, 'label': 10}
{'start': 637, 'end': 644, 'label': 5}
{'start': 695, 'end': 704, 'label': 5}
{'start': 792, 'end': 800, 'label': 0}
{'start': 1122, 'end': 1130, 'label': 0}
{'start': 1341, 'end': 1349, 'label': 0}
{'start': 1373, 'end': 1380, 'label': 5}
{'start': 1583, 'end': 1590, 'label': 5}
{'start': 1809, 'end': 1822, 'label': 5}
{'start': 2229, 'end': 2242, 'label': 5}
{'start': 2296, 'end': 2303, 'label': 5}
{'start': 2314, 'end': 2323, 'label': 5}
{'start': 2341, 'end': 2348, 'label': 5}
{'start': 2491, 'end': 2494, 'label': 8}
{'start': 2501, 'end': 2513, 'label': 5}
{'start': 2685, 'end': 2688, 'label': 8}
{'start': 2756, 'end': 2763, 'label': 5}
{'start': 2769, 'end': 2778, 'label': 5}
{'start': 2791, 'end': 2801, 'label': 5}
{'start': 2814, 'end': 2829, 'label': 5}
{'start': 2941, 'end': 2948, '

#Step 3

In [ ]:
from transformers import AutoTokenizer
from datasets import DatasetDict
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Use fold_1 for baseline
train_ds = ds["fold_1"]

# Extract unique labels from the dataset and create mappings
label_names = sorted(list(set([str(span["label"]) for example in ds["fold_1"] for span in example["spans"]])))
# Add "O" label at the beginning
label2id = {label: i for i, label in enumerate(["O"] + label_names)}
id2label = {i: label for label, i in label2id.items()}

print(f"Labels in the dataset: {label_names}")
print(f"ID to label mapping: {id2label}")
print(f"Label to ID mapping: {label2id}")


def align_labels_with_tokens(examples):
    texts = examples["text"]
    all_labels = []

    for text, spans in zip(texts, examples["spans"]):
        # Initialize all chars as 'O'
        char_labels = ["O"] * len(text)
        for span in spans:
            start, end, label = span["start"], span["end"], span["label"]
            # Convert label to string before concatenation
            char_labels[start:end] = ["I-" + str(label)] * (end - start)
            if end > start:
                char_labels[start] = "B-" + str(label)

        # Tokenize text
        tokenized = tokenizer(text, truncation=True, max_length=512)
        word_ids = tokenized.word_ids(batch_index=0)

        labels = []
        for i, word_id in enumerate(word_ids):
            if word_id is None:
                labels.append(-100)  # ignore in loss
            else:
                token_start = tokenized.token_to_chars(i).start
                label = char_labels[token_start] if token_start < len(char_labels) else "O"
                # Convert string label to integer ID using label2id
                labels.append(label2id.get(label, label2id["O"])) # Use .get() with default for safety

        all_labels.append(labels)

    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=512)
    tokenized["labels"] = all_labels
    return tokenized

# Apply mapping
tokenized_ds = train_ds.map(align_labels_with_tokens, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["id", "text", "spans"])
tokenized_ds.set_format("torch")

print(tokenized_ds)

/usr/local/lib/python3.12/dist-packages/torch_xla/experimental/gru.py:113: SyntaxWarning: invalid escape sequence '\_'
  * **h_n**: tensor of shape :math:`(D * \text{num\_layers}, H_{out})` or


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Labels in the dataset: ['0', '1', '10', '11', '2', '3', '4', '5', '6', '7', '8', '9']
ID to label mapping: {0: 'O', 1: '0', 2: '1', 3: '10', 4: '11', 5: '2', 6: '3', 7: '4', 8: '5', 9: '6', 10: '7', 11: '8', 12: '9'}
Label to ID mapping: {'O': 0, '0': 1, '1': 2, '10': 3, '11': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12}


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 35
})


In [ ]:
from transformers import AutoTokenizer
from datasets import DatasetDict
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Use fold_1 for baseline
train_ds = ds["fold_1"]

# Label ID → Entity name mapping (IL-TUR official) - Define label_map here as the source of truth
label_map = {
    0: "COURT",
    1: "PETITIONER",
    2: "RESPONDENT",
    3: "STATUTE",
    4: "PROVISION",
    5: "PRECEDENT",
    6: "CASE_NUMBER",
    7: "DATE",
    8: "OTHER_PERSON",
    9: "ORGANISATION",
    10: "GPE"
}


# Create label name -> ID mapping, including 'O', B- and I- tags
# Get unique label names from the map
unique_label_names = sorted(list(label_map.values()))

# Create label2id mapping
label2id = {"O": 0}
for i, label_name in enumerate(unique_label_names):
    label2id[f"B-{label_name}"] = 2 * i + 1
    label2id[f"I-{label_name}"] = 2 * i + 2

# Create id2label mapping
id2label = {id: label for label, id in label2id.items()}
label_list = list(label2id.keys()) # Define label_list

print(f"Generated label_list: {label_list}")
print(f"Size of label_list: {len(label_list)}")
print(f"ID to label mapping: {id2label}")

def align_labels_with_tokens(examples):
    texts = examples["text"]
    all_labels = []

    tokenized_inputs = tokenizer(texts, truncation=True, padding="max_length", max_length=512) # Add padding here

    for i, spans in enumerate(examples["spans"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = [-100] * len(word_ids)

        char_labels = ["O"] * len(texts[i]) # Initialize char_labels for the current text
        for span in spans:
            start, end, label = span["start"], span["end"], span["label"]
            # Convert label to string before concatenation
            char_labels[start:end] = ["I-" + str(label_map.get(label, "O"))] * (end - start) # Use label_map to get name
            if end > start:
                char_labels[start] = "B-" + str(label_map.get(label, "O")) # Use label_map to get name


        for j, word_id in enumerate(word_ids):
            if word_id is None:
                label_ids[j] = -100  # ignore in loss
            else:
                token_start = tokenized_inputs.token_to_chars(i, j).start # Get token start char index
                label = char_labels[token_start] if token_start < len(char_labels) else "O"
                # Convert string label to integer ID using label2id
                label_ids[j] = label2id.get(label, label2id["O"]) # Use .get() with default for safety

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Apply mapping
tokenized_ds = train_ds.map(align_labels_with_tokens, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["id", "text", "spans"])
tokenized_ds.set_format("torch")

print(tokenized_ds)

Generated label_list: ['O', 'B-CASE_NUMBER', 'I-CASE_NUMBER', 'B-COURT', 'I-COURT', 'B-DATE', 'I-DATE', 'B-GPE', 'I-GPE', 'B-ORGANISATION', 'I-ORGANISATION', 'B-OTHER_PERSON', 'I-OTHER_PERSON', 'B-PETITIONER', 'I-PETITIONER', 'B-PRECEDENT', 'I-PRECEDENT', 'B-PROVISION', 'I-PROVISION', 'B-RESPONDENT', 'I-RESPONDENT', 'B-STATUTE', 'I-STATUTE']
Size of label_list: 23
ID to label mapping: {0: 'O', 1: 'B-CASE_NUMBER', 2: 'I-CASE_NUMBER', 3: 'B-COURT', 4: 'I-COURT', 5: 'B-DATE', 6: 'I-DATE', 7: 'B-GPE', 8: 'I-GPE', 9: 'B-ORGANISATION', 10: 'I-ORGANISATION', 11: 'B-OTHER_PERSON', 12: 'I-OTHER_PERSON', 13: 'B-PETITIONER', 14: 'I-PETITIONER', 15: 'B-PRECEDENT', 16: 'I-PRECEDENT', 17: 'B-PROVISION', 18: 'I-PROVISION', 19: 'B-RESPONDENT', 20: 'I-RESPONDENT', 21: 'B-STATUTE', 22: 'I-STATUTE'}


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 35
})


# Step 4 — Build and train a baseline NER model

In [ ]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Define label list from the earlier mapping
label_list = ["O"] + [f"B-{v}" for v in label_map.values()] + [f"I-{v}" for v in label_map.values()]
num_labels = len(label_list)

# Label ↔ id mapping
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Split dataset into train & validation (since only 35 rows)
train_test = tokenized_ds.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test["train"]
eval_dataset = train_test["test"]

# Metric for evaluation
metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Training arguments
args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Corrected argument name
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",  # disables wandb
    optim="adamw_torch" # Explicitly set optimizer to avoid fused Adam with XLA
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Train the model 🚀
trainer.train()

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


/tmp/ipython-input-1086106230.py:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.410034,0.000000,0.000000,0.000000,0.864426
2,1.951100,0.779348,0.000000,0.000000,0.000000,0.864426
3,0.890300,0.732439,0.000000,0.000000,0.000000,0.864426


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `z

TrainOutput(global_step=21, training_loss=1.3908792194866, metrics={'train_runtime': 144.5164, 'train_samples_per_second': 0.581, 'train_steps_per_second': 0.145, 'total_flos': 21953094782976.0, 'train_loss': 1.3908792194866, 'epoch': 3.0})

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load tokenizer and model again (use the same model name as in your earlier code)
model_name = "dslim/bert-base-NER"  # or replace with your fine-tuned model if you trained one
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Move to CUDA
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

# Sample sentence for test
text = "The case was heard in the Supreme Court and filed by ABC Pvt Ltd under Article 32."

# Tokenize and predict
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, device=0 if device=="cuda:0" else -1, aggregation_strategy="simple")

result = ner_pipeline(text)
print(result)

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'entity_group': 'ORG', 'score': np.float32(0.99766827), 'word': 'Supreme Court', 'start': 26, 'end': 39}, {'entity_group': 'ORG', 'score': np.float32(0.99736357), 'word': 'ABC Pvt Ltd', 'start': 53, 'end': 64}]


# Fine-tune Legal-BERT for IL-TUR NER

In [ ]:
!pip install -q transformers datasets accelerate seqeval evaluate
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import concatenate_datasets
import evaluate
import numpy as np


In [ ]:

model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
# Combine all folds
full_dataset = concatenate_datasets([ds['fold_1'], ds['fold_2'], ds['fold_3']])

# Tokenize & align labels
tokenized_full = full_dataset.map(align_labels_with_tokens, batched=True)
tokenized_full = tokenized_full.remove_columns(["id", "text", "spans"])
tokenized_full.set_format("torch")

# Train / validation split
train_test = tokenized_full.train_test_split(test_size=0.15, seed=42)
train_dataset = train_test["train"]
eval_dataset = train_test["test"]


Map:   0%|          | 0/105 [00:00<?, ? examples/s]

In [ ]:

metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
args = TrainingArguments(
    output_dir="./legal_bert_ner_results",
    eval_strategy="epoch", # Corrected argument name
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,  # train longer for better results
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    optim="adamw_torch" # Explicitly set optimizer to avoid fused Adam with XLA
)

In [ ]:
from transformers import Trainer, DataCollatorForTokenClassification

# Initialize trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer), # Explicitly use DataCollatorForTokenClassification
    compute_metrics=compute_metrics
)

# Fine-tune Legal-BERT
trainer.train()

/tmp/ipython-input-520463470.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.940000,0.835170,0.000000,0.000000,0.000000,0.835539
2,0.639900,0.615807,0.016129,0.004082,0.006515,0.853922
3,0.525900,0.523541,0.272727,0.208163,0.236111,0.871936
4,0.400200,0.495484,0.278261,0.261224,0.269474,0.875368
5,0.369400,0.482803,0.283333,0.277551,0.280412,0.879534


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be us

TrainOutput(global_step=115, training_loss=0.668289219814798, metrics={'train_runtime': 243.2367, 'train_samples_per_second': 1.891, 'train_steps_per_second': 0.473, 'total_flos': 116299133076480.0, 'train_loss': 0.668289219814798, 'epoch': 5.0})

# Step 5 — Run inference with fine-tuned Legal-BERT

In [ ]:
from transformers import pipeline

# NER pipeline using fine-tuned model
ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,  # uses CUDA
    aggregation_strategy="simple"
)

# Sample legal text
text = "The Supreme Court of India dismissed the petition filed by ABC Pvt Ltd under Article 32 of the Constitution."

# Run NER
preds = ner_pipeline(text)

# Display results
for p in preds:
    print(p)


Device set to use cpu


In [ ]:
# Legal entity labels (IL-TUR)
entity_labels = ["O"] + [f"B-{v}" for v in label_map.values()] + [f"I-{v}" for v in label_map.values()]

# Create id ↔ label mapping
id2label = {i: label for i, label in enumerate(entity_labels)}
label2id = {label: i for i, label in enumerate(entity_labels)}

# Update model
model.config.id2label = id2label
model.config.label2id = label2id


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# Tokenize
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model(**inputs)
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)[0].cpu().numpy()

# Map token predictions to labels
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
pred_labels = [id2label[p] for p in predictions]

# Show tokens with predicted labels
for token, label in zip(tokens, pred_labels):
    if label != "O":  # show only entities
        print(token, "→", label)


[SEP] → I-OTHER_PERSON


# Demo NER with visible legal entities

In [ ]:
from transformers import pipeline

# Load general pretrained NER
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=0  # use GPU
)

# Sample legal text
text = "The Supreme Court of India dismissed the petition filed by ABC Pvt Ltd under Article 32 of the Constitution."

# Run NER
preds = ner_pipeline(text)

# Map pretrained labels to IL-TUR legal entities (simple example)
label_mapping = {
    "ORG": "PETITIONER/COURT",
    "LOC": "COURT",
    "PER": "PETITIONER"
}

# Display mapped entities
for p in preds:
    iltur_label = label_mapping.get(p['entity_group'], p['entity_group'])
    print(f"{p['word']} → {iltur_label}, score: {p['score']:.2f}")


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Supreme Court of India → PETITIONER/COURT, score: 1.00
ABC Pvt Ltd → PETITIONER/COURT, score: 1.00
Constitution → MISC, score: 0.78


In [ ]:
from spacy import displacy

# Convert predictions to Spacy format
ents = []
for p in preds:
    ents.append({
        "start": p['start'],
        "end": p['end'],
        "label": label_mapping.get(p['entity_group'], p['entity_group'])
    })

doc = {"text": text, "ents": ents, "title": None}

# Render in notebook
displacy.render(doc, style="ent", manual=True)
